In [1]:
!pip install pyspark tensorflow tensorflow-lite scikit-learn

ERROR: Could not find a version that satisfies the requirement tensorflow-lite (from versions: none)
ERROR: No matching distribution found for tensorflow-lite


In [2]:
!pip install pyspark

In [3]:
pip install ai-edge-litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 64.3 MB/s eta 0:00:00


In [4]:
!pip install pyspark tensorflow scikit-learn

In [5]:
!pip install pyspark tensorflow scikit-learn tflite-model-maker

INFO: pip is looking at multiple versions of tflite-model-maker to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 1.6 MB/s eta 0:00:00
Requested tflite-model-maker from https://files.pythonhosted.org/packages/29/dc/f74a8aa41a8b78dc57e6cf1acf0003b40753c8b73ce581532ceff1442aaf/tflite_model_maker-0.3.2-py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    tensorflow-hub (<0.10>=0.8.0)
                   ~~~~~~^
Please use pip<24.1 if you need to use this version.
Requested tflite-model-maker from https://files.pythonhosted.org/packages/ea/96/b3149af45895426a918591d57a237323dbc2d331abb0d27d1adb9e218995/tflite_model_maker-0.3.1-py3-none-any.whl has invalid metadata: Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier
    tensorflow-hub (<0.10>=0.8.0)
                   ~~~~~~^
Please use pip<2

In [6]:
!pip install pyspark tensorflow scikit-learn tflite-model-maker

  Using cached tflite_model_maker-0.4.3-py3-none-any.whl.metadata (5.4 kB)
  Using cached tf_models_official-2.3.0-py2.py3-none-any.whl.metadata (1.3 kB)
INFO: pip is looking at multiple versions of tflite-model-maker to determine which version is compatible with other requirements. This could take a while.
  Using cached tflite_model_maker-0.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached fire-0.7.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached urllib3-1.25.11-py2.py3-none-any.whl.metadata (41 kB)
  Using cached tflite_model_maker-0.4.1-py3-none-any.whl.metadata (5.4 kB)
  Using cached tflite_model_maker-0.4.0-py3-none-any.whl.metadata (5.5 kB)
  Using cached tflite_model_maker-0.3.4-py3-none-any.whl.metadata (5.4 kB)
  Using cached tflite_model_maker-0.3.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached tflite_model_maker-0.3.2-py3-none-any.whl.metadata (5.3 kB)
Requested tflite-model-maker from https://files.pythonhosted.org/packages/29/dc/f74a8aa41a8b78dc57e6cf1acf0003b

In [7]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import tensorflow as tf
import time

In [8]:
spark = SparkSession.builder.appName("WaterPurity").getOrCreate()

In [9]:
df = spark.read.csv(
"/content/water_quality_data.csv",
header=True,
inferSchema=True
)

In [10]:
df.show(5)

+-------+-----------+--------+--------------------+-------------------+-----------+--------+--------+-----------+-----------+-----+---+-----------------------+---------+--------------+---------------------+----------------+-------------+--------------+----------------+------------------+--------+--------+--------+--------+----------+----------+------------+--------------+
| region|    village|district|         source_name|lab_identifier_code|source_type| easting|northing|   latitude|  longitude|ecoli| ph|electrical_conductivity|turbidity|color_apparent|total_dissolved_salts|total_alkalinity|bi_carbonates|total_hardness|calcium_hardness|magnesium_hardness|flouride|chloride|sulphate|nitrites|nitrates_n|ammonium_n|phosphates_p| water_quality|
+-------+-----------+--------+--------------------+-------------------+-----------+--------+--------+-----------+-----------+-----+---+-----------------------+---------+--------------+---------------------+----------------+-------------+-------------

In [11]:
df.printSchema()

root
 |-- region: string (nullable = true)
 |-- village: string (nullable = true)
 |-- district: string (nullable = true)
 |-- source_name: string (nullable = true)
 |-- lab_identifier_code: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- easting: double (nullable = true)
 |-- northing: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- ecoli: integer (nullable = true)
 |-- ph: double (nullable = true)
 |-- electrical_conductivity: double (nullable = true)
 |-- turbidity: double (nullable = true)
 |-- color_apparent: double (nullable = true)
 |-- total_dissolved_salts: double (nullable = true)
 |-- total_alkalinity: double (nullable = true)
 |-- bi_carbonates: double (nullable = true)
 |-- total_hardness: double (nullable = true)
 |-- calcium_hardness: double (nullable = true)
 |-- magnesium_hardness: double (nullable = true)
 |-- flouride: double (nullable = true)
 |-- chloride: double (nullable = tru

In [12]:
df= df.dropna()

In [13]:
pdf= df.toPandas()

In [14]:
pdf.head()

,region,village,district,source_name,lab_identifier_code,source_type,easting,northing,latitude,longitude,...,calcium_hardness,magnesium_hardness,flouride,chloride,sulphate,nitrites,nitrates_n,ammonium_n,phosphates_p,water_quality
0,Eastern,Bugembe,Jinja,Kilikwani tap,E23/00103,Tap,526362.0,51478.0,0.465734,33.236916,...,18.0,0.0,0.26236,0.0,4.10009,0.01509,0.32250,0.16075,0.03779,not admissible
1,Eastern,Masese,Jinja,Ndiba Roseline tap,E23/00104,Tap,526349.0,49702.0,0.449666,33.236798,...,19.0,0.0,0.22605,0.0,3.72661,0.00318,2.49162,0.00200,0.09726,not admissible
2,Eastern,Masese,Jinja,Ndiba Roseline tap,E23/00104,Tap,526921.0,48543.0,0.449666,33.236798,...,19.0,0.0,0.25573,0.0,4.27461,0.00760,0.51466,0.03963,0.05778,not admissible
3,Eastern,Rubaga hill,Jinja,Jinja college Borehole,E23/00109,Bore Hole,523658.0,49584.0,0.448599,33.212614,...,125.0,0.0,0.66970,0.0,69.27158,0.00678,6.68600,0.00200,0.99905,not admissible
4,Eastern,Mpumudde,Jinja,Mpumudde mosque B/H,E22/00110,Bore Hole,523831.0,51559.0,0.466467,33.214170,...,230.0,0.0,0.68100,0.0,62.21922,0.08142,17.75154,0.02197,0.05519,not admissible


In [15]:
X = pdf.drop("water_quality", axis=1)
y = pdf["water_quality"]

In [16]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

In [21]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.model_selection import train_test_split

# Re-split X and y to ensure X_train and X_test are DataFrames for initial processing.
# This makes the cell robust to re-execution.
X_train_df, X_test_df, _, _ = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify categorical columns
categorical_cols = X_train_df.select_dtypes(include=['object']).columns

# Apply One-Hot Encoding
X_train_encoded = pd.get_dummies(X_train_df, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_df, columns=categorical_cols, drop_first=True)

# Align columns after one-hot encoding to ensure X_test has the same columns as X_train
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# Initialize and apply StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_encoded)
X_test = scaler.transform(X_test_encoded)

In [22]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)), # Explicit Input layer
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1)
])

In [23]:
model.compile(
optimizer="adam",
loss="mse",
metrics=["mae"]
)

In [25]:
from sklearn.preprocessing import LabelEncoder

# Encode y_train to numerical labels if it contains object (string) types
if y_train.dtype == 'object':
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
else:
    y_train_encoded = y_train

model.fit(
    X_train,
    y_train_encoded, # Use the numerically encoded target variable
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 105ms/step - loss: 2.3745 - mae: 1.3262 - val_loss: 1.4597 - val_mae: 0.9788
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.6408 - mae: 0.6127 - val_loss: 1.1039 - val_mae: 0.8470
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.4099 - mae: 0.5244 - val_loss: 1.0270 - val_mae: 0.8265
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.3414 - mae: 0.4992 - val_loss: 1.0045 - val_mae: 0.8055
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.1592 - mae: 0.3346 - val_loss: 1.0170 - val_mae: 0.7801
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.0524 - mae: 0.1745 - val_loss: 1.0562 - val_mae: 0.7901
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 0.0543 - mae: 0.1684 - val_loss: 1.0830 - val_mae: 0.8075
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0597 - mae: 0.1866 - val_loss: 1.0776 - val_mae: 0.8058
Epoch 9/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - loss: 0.0337 - mae: 0.1404 -

In [27]:
# Encode y_test to numerical labels using the fitted LabelEncoder
if y_test.dtype == 'object':
    # Use the 'le' LabelEncoder fitted in the previous cell
    y_test_encoded = le.transform(y_test)
else:
    y_test_encoded = y_test

model.evaluate(X_test, y_test_encoded)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 0.1760 - mae: 0.3021


[0.17600545287132263, 0.30206912755966187]

In [28]:
model.save("cloud_WaterQuality_model.keras")

In [29]:
model.export("cloud_WaterQuality_model")

Saved artifact at 'cloud_WaterQuality_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 377), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132170944835472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132170944836624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132170944835280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132170944833552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132170944837200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132170944832976: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [30]:
model.save("cloud_WaterQuality_model.h5")

In [33]:
converter = tf.lite.TFLiteConverter.from_saved_model("cloud_WaterQuality_model")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open("purity_edge_model.tflite", "wb") as f: f.write(tflite_model)

In [34]:
import os
print("Cloud model size:",
os.path.getsize("cloud_WaterQuality_model/saved_model.pb")/1024,
"KB")
print("Edge model size:",
os.path.getsize("purity_edge_model.tflite")/1024,
"KB")

Cloud model size: 42.7314453125 KB
Edge model size: 29.0625 KB


In [35]:
interpreter = tf.lite.Interpreter(model_path="purity_edge_model.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [36]:
!pip install ai-edge-litert

In [37]:
import ai_edge_litert as litert

# Use the LiteRT interpreter instead
interpreter = tf.lite.Interpreter(model_path="housing_edge_model.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [38]:
sample = np.array([X_test[0]], dtype=np.float32)
interpreter.set_tensor(input_details[0]['index'], sample)
start = time.time()
interpreter.invoke()
prediction = interpreter.get_tensor(output_details[0]['index'])
end = time.time()
print("Water Quality:", prediction[0][0])
print("Edge inference time:", end-start)

Water Quality: 0.82620394
Edge inference time: 0.0026907920837402344


In [39]:
start = time.time()
cloud_pred = model.predict(sample)
end = time.time()
print("Cloud prediction:", cloud_pred[0][0])
print("Cloud inference time:", end-start)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
Cloud prediction: 0.80501527
Cloud inference time: 0.23204898834228516


In [41]:
edge_times = []
cloud_times = []

# Iterate over the actual number of samples in X_test
for i in range(len(X_test)):
    sample = np.array([X_test[i]], dtype=np.float32)

    # Edge inference
    start = time.time()
    interpreter.set_tensor(input_details[0]['index'], sample)
    interpreter.invoke()
    interpreter.get_tensor(output_details[0]['index'])
    edge_times.append(time.time()-start)

    # Cloud inference
    start = time.time()
    model.predict(sample, verbose=0)
    cloud_times.append(time.time()-start)

print("Average Edge Latency:", np.mean(edge_times))
print("Average Cloud Latency:", np.mean(cloud_times))

Average Edge Latency: 0.00011369254853990342
Average Cloud Latency: 0.20064054595099556


In [44]:
import os
import numpy as np
import time

edge_times = []
cloud_times = []

# 1. Bandwidth (Payload Size Calculation)
# Estimate size of data sent over network (input + output) in Kilobytes
# Initialize sample correctly before calculating payload_kb
if len(X_test) > 0:
    sample_for_payload = np.array([X_test[0]], dtype=np.float32)
    payload_kb = (sample_for_payload.nbytes + np.array([0.0], dtype=np.float32).nbytes) / 1024
else:
    payload_kb = 0.0 # Handle case where X_test might be empty

# Iterate over the actual number of samples in X_test
for i in range(len(X_test)):
    sample = np.array([X_test[i]], dtype=np.float32)

    # --- Edge Inference ---
    start = time.time()
    interpreter.set_tensor(input_details[0]['index'], sample)
    interpreter.invoke()
    interpreter.get_tensor(output_details[0]['index'])
    edge_times.append(time.time() - start)

    # --- Cloud Inference ---
    start = time.time()
    model.predict(sample, verbose=0)
    cloud_times.append(time.time() - start)

# Results Comparison
print(f"--- Benchmarking Results (N={len(X_test)}) ---")
print(f"Average Edge Latency: {np.mean(edge_times):.5f}s")
print(f"Average Cloud Latency: {np.mean(cloud_times):.5f}s")

print(f"\n--- Edge vs Cloud Parameters ---")
# Bandwidth: Edge uses 0kb of network traffic
print(f"Bandwidth Usage (Cloud): {payload_kb:.2f} KB/request")
print(f"Bandwidth Usage (Edge): 0.00 KB (Local processing)")

# Privacy: Data remains on device for Edge
print(f"Privacy (Cloud): Lower (Data transmitted to external server)")
print(f"Privacy (Edge): Higher (Data never leaves the device)")

# Offline Environment: Is it functional?
print(f"Offline Environment (Cloud): No (Requires active connection)")
print(f"Offline Environment (Edge): Yes (Fully functional offline)")

--- Benchmarking Results (N=36) ---
Average Edge Latency: 0.00008s
Average Cloud Latency: 0.21283s

--- Edge vs Cloud Parameters ---
Bandwidth Usage (Cloud): 1.48 KB/request
Bandwidth Usage (Edge): 0.00 KB (Local processing)
Privacy (Cloud): Lower (Data transmitted to external server)
Privacy (Edge): Higher (Data never leaves the device)
Offline Environment (Cloud): No (Requires active connection)
Offline Environment (Edge): Yes (Fully functional offline)


In [47]:
feedback_data = []
feedback_labels = []

# Iterate only over the available samples in X_test
for i in range(len(X_test)):
    pred = model.predict(np.array([X_test[i]]), verbose=0)[0][0]
    # Use the numerically encoded y_test value
    true = y_test_encoded[i]

    # Use a more reasonable error threshold for binary classification (0 or 1)
    if abs(pred - true) > 0.5:
        feedback_data.append(X_test[i])
        feedback_labels.append(true)

print("Feedback samples collected:", len(feedback_data))

Feedback samples collected: 5


In [49]:
import numpy as np

# Configuration
TEST_RANGE = len(X_test) # Adjust TEST_RANGE to the actual number of samples in X_test
ERROR_THRESHOLD = 0.5 # Adjusted to a more reasonable value for binary classification

print(f"--- Starting Distributed Feedback Analysis (Range: {TEST_RANGE}) ---\n")

# Efficient Batch Prediction
X_sample = X_test[:TEST_RANGE]
y_true = y_test_encoded[:TEST_RANGE] # Use the numerically encoded target variable
predictions = model.predict(X_sample, verbose=0).flatten()

feedback_data = []
feedback_labels = []

for i in range(TEST_RANGE):
    error = abs(predictions[i] - y_true[i])
    if error > ERROR_THRESHOLD:
        feedback_data.append(X_sample[i])
        feedback_labels.append(y_true[i])

# --- Automated Explanations ---
failure_rate = (len(feedback_data) / TEST_RANGE) * 100

print(f"ANALYSIS COMPLETE:")
print(f"1. Samples Analyzed: {TEST_RANGE} (Representing a broad slice of the water quality data)")
print(f"2. Errors Detected: {len(feedback_data)} instances where the model missed by > {ERROR_THRESHOLD}")
print(f"3. Failure Rate: {failure_rate:.2f}%")

print("\n--- DISTRIBUTED ML INTERPRETATION ---")
if failure_rate > 20:
    print("STATUS: [HIGH ERROR RATE]")
    print("EXPLANATION: The model is struggling with global generalization. This large batch of feedback data is CRITICAL for retraining.")
elif failure_rate > 5:
    print("STATUS: [MODERATE ERROR RATE]")
    print("EXPLANATION: These samples likely represent 'Edge Cases' (e.g., specific water conditions). Sending these back to the cloud will refine the model's precision.")
else:
    print("STATUS: [LOW ERROR RATE]")
    print("EXPLANATION: The model is highly stable. Very little data needs to be sent back to the cloud, saving significant network bandwidth.")

print(f"\nACTION: Collected {len(feedback_data)} 'Hard Examples' for the Cloud retraining queue.")

--- Starting Distributed Feedback Analysis (Range: 36) ---

ANALYSIS COMPLETE:
1. Samples Analyzed: 36 (Representing a broad slice of the water quality data)
2. Errors Detected: 5 instances where the model missed by > 0.5
3. Failure Rate: 13.89%

--- DISTRIBUTED ML INTERPRETATION ---
STATUS: [MODERATE ERROR RATE]
EXPLANATION: These samples likely represent 'Edge Cases' (e.g., specific water conditions). Sending these back to the cloud will refine the model's precision.

ACTION: Collected 5 'Hard Examples' for the Cloud retraining queue.


In [53]:
!pip install streamlit
import streamlit as st
import pandas as pd
import plotly.express as px
import numpy as np
import tensorflow as tf # Import tensorflow for loading Keras model

# Load the Keras model directly
@st.cache_resource
def load_model():
    return tf.keras.models.load_model('cloud_WaterQuality_model.keras')

model = load_model()

st.set_page_config(page_title=": Water Quality Intelligence", layout="wide")
st.markdown("""
    <style>
    .main {background-color: #0e1117;}
    .sidebar .sidebar-content {background-color: #1a1d2e;}
    .metric {color: #00d4aa;}
    </style>
""", unsafe_allow_html=True)

st.title("🧪 : Regional Water Quality Intelligence")
st.markdown("**IoT Monitoring & ML Predictions for Kenya Lakes/Rivers**")

# Sidebar (Sensors)
st.sidebar.header("🌡️ IoT Sensors")
ph_input = st.sidebar.slider("pH", 0.0, 14.0, 7.0)
cond_input = st.sidebar.slider("Conductivity (µS/cm)", 0.0, 2000.0, 200.0)
turb_input = st.sidebar.slider("Turbidity (NTU)", 0.0, 100.0, 5.0)
nit_input = st.sidebar.slider("Nitrates (mg/L)", 0.0, 50.0, 10.0)

# --- Preprocess single input for model prediction ---
# Assuming X_train_encoded, categorical_cols, and scaler are available in the global scope
# from previous notebook cells where they were defined and fitted.

# Create an empty DataFrame with the same columns as X_train_encoded (377 features)
# This ensures the input has the correct structure for the model.
input_df_single = pd.DataFrame(0, index=[0], columns=X_train_encoded.columns)

# Populate the known values from the Streamlit sliders
input_df_single['ph'] = ph_input
input_df_single['electrical_conductivity'] = cond_input
input_df_single['turbidity'] = turb_input
input_df_single['nitrates_n'] = nit_input

# Scale the input data using the pre-fitted scaler
# The 'scaler' object is available from the kernel state.
processed_input_data = scaler.transform(input_df_single)

# Make prediction using the correctly shaped and scaled input
pred = model.predict(processed_input_data)[0]
quality = "Safe 🟢" if pred == 1 else "Unsafe 🔴"

# Main Metrics
col1, col2, col3, col4 = st.columns(4)
col1.metric("pH", f"{ph_input:.1f}", delta="Stable")
col2.metric("Turbidity", f"{turb_input:.1f} NTU", delta="Low")
col3.metric("Predicted Quality", quality, delta="Real-time")
col4.metric("Alert Level", "Low", delta="No issues")

# Chart
df_sample = pd.DataFrame({
    'pH': [ph_input, 6.5, 8.0],
    'Turbidity': [turb_input, 10, 20],
    'Quality': ['Predicted', 'Safe', 'Unsafe']
})
fig = px.scatter(df_sample, x='pH', y='Turbidity', color='Quality', title="Water Quality Scatter")
st.plotly_chart(fig, use_container_width=True)

# Upload CSV for batch predict
uploaded = st.file_uploader("Upload Kenya Water CSV")
if uploaded:
    df_up = pd.read_csv(uploaded)

    # --- Preprocess batch input for model prediction ---
    # Assuming categorical_cols and scaler are available in the global scope.

    # Identify categorical columns from the uploaded DataFrame (if any new ones)
    # Or, preferably, use the 'categorical_cols' identified during training
    # For robust processing, we re-identify categorical columns from the uploaded df
    # and then align with the training columns.
    uploaded_categorical_cols = df_up.select_dtypes(include=['object']).columns

    # Apply One-Hot Encoding to the uploaded data
    df_up_encoded = pd.get_dummies(df_up, columns=uploaded_categorical_cols, drop_first=True)

    # Align columns after one-hot encoding to ensure df_up_encoded has the same columns as X_train_encoded
    # Fill with 0 for columns present in X_train_encoded but not in df_up_encoded
    df_up_encoded = df_up_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

    # Scale the uploaded data using the fitted scaler
    df_up_scaled = scaler.transform(df_up_encoded)

    # Make batch predictions
    df_up['prediction'] = model.predict(df_up_scaled).flatten()
    st.dataframe(df_up)

2026-03-25 04:50:14.326 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.327 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.330 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.333 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.336 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.340 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.344 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.346 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step


2026-03-25 04:50:14.731 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.732 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.734 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.736 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.742 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.745 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 04:50:14.746 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [55]:
%%writefile water_quality_app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import numpy as np
import tensorflow as tf # Import tensorflow for loading Keras model

# Load the Keras model directly
@st.cache_resource
def load_model():
    return tf.keras.models.load_model('cloud_WaterQuality_model.keras')

model = load_model()

st.set_page_config(page_title=": Water Quality Intelligence", layout="wide")
st.markdown("""
    <style>
    .main {background-color: #0e1117;}
    .sidebar .sidebar-content {background-color: #1a1d2e;}
    .metric {color: #00d4aa;}
    </style>
""", unsafe_allow_html=True)

st.title("🧪 : Regional Water Quality Intelligence")
st.markdown("**IoT Monitoring & ML Predictions for Kenya Lakes/Rivers**")

# Sidebar (Sensors)
st.sidebar.header("🌡️ IoT Sensors")
ph_input = st.sidebar.slider("pH", 0.0, 14.0, 7.0)
cond_input = st.sidebar.slider("Conductivity (µS/cm)", 0.0, 2000.0, 200.0)
turb_input = st.sidebar.slider("Turbidity (NTU)", 0.0, 100.0, 5.0)
nit_input = st.sidebar.slider("Nitrates (mg/L)", 0.0, 50.0, 10.0)

# --- Preprocess single input for model prediction ---
# Assuming X_train_encoded, categorical_cols, and scaler are available in the global scope
# from previous notebook cells where they were defined and fitted.

# Create an empty DataFrame with the same columns as X_train_encoded (377 features)
# This ensures the input has the correct structure for the model.
input_df_single = pd.DataFrame(0, index=[0], columns=X_train_encoded.columns)

# Populate the known values from the Streamlit sliders
input_df_single['ph'] = ph_input
input_df_single['electrical_conductivity'] = cond_input
input_df_single['turbidity'] = turb_input
input_df_single['nitrates_n'] = nit_input

# Scale the input data using the pre-fitted scaler
# The 'scaler' object is available from the kernel state.
processed_input_data = scaler.transform(input_df_single)

# Make prediction using the correctly shaped and scaled input
pred = model.predict(processed_input_data)[0]
quality = "Safe 🟢" if pred == 1 else "Unsafe 🔴"

# Main Metrics
col1, col2, col3, col4 = st.columns(4)
col1.metric("pH", f"{ph_input:.1f}", delta="Stable")
col2.metric("Turbidity", f"{turb_input:.1f} NTU", delta="Low")
col3.metric("Predicted Quality", quality, delta="Real-time")
col4.metric("Alert Level", "Low", delta="No issues")

# Chart
df_sample = pd.DataFrame({
    'pH': [ph_input, 6.5, 8.0],
    'Turbidity': [turb_input, 10, 20],
    'Quality': ['Predicted', 'Safe', 'Unsafe']
})
fig = px.scatter(df_sample, x='pH', y='Turbidity', color='Quality', title="Water Quality Scatter")
st.plotly_chart(fig, use_container_width=True)

# Upload CSV for batch predict
uploaded = st.file_uploader("Upload Kenya Water CSV")
if uploaded:
    df_up = pd.read_csv(uploaded)

    # --- Preprocess batch input for model prediction ---
    # Assuming categorical_cols and scaler are available in the global scope.

    # Identify categorical columns from the uploaded DataFrame (if any new ones)
    # Or, preferably, use the 'categorical_cols' identified during training
    # For robust processing, we re-identify categorical columns from the uploaded df
    # and then align with the training columns.
    uploaded_categorical_cols = df_up.select_dtypes(include=['object']).columns

    # Apply One-Hot Encoding to the uploaded data
    df_up_encoded = pd.get_dummies(df_up, columns=uploaded_categorical_cols, drop_first=True)

    # Align columns after one-hot encoding to ensure df_up_encoded has the same columns as X_train_encoded
    # Fill with 0 for columns present in X_train_encoded but not in df_up_encoded
    df_up_encoded = df_up_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

    # Scale the uploaded data using the fitted scaler
    df_up_scaled = scaler.transform(df_up_encoded)

    # Make batch predictions
    df_up['prediction'] = model.predict(df_up_scaled).flatten()
    st.dataframe(df_up)

Overwriting water_quality_app.py


In [58]:
!streamlit run water_quality_app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.181.224:8501

  Stopping...
  Stopping...
Traceback (most recent call last):
  File "/usr/local/bin/streamlit", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1485, in __call__
    return self.main(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1406, in main
    rv = self.invoke(ctx)
         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1873, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/click/core.py", line 1269, in invoke
    return ctx.invoke(self.callback, **ctx.params)
           ^^^^^^^^^^^^^^^

In [60]:
!streamlit run water_quality_app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.181.224:8501

  Stopping...
  Stopping...
  Stopping...


In [61]:
!pip -q install streamlit pyngrok joblib plotly pandas scikit-learn
!pip -q install nest-asyncio  # For threading
import nest_asyncio
nest_asyncio.apply()


In [62]:
%%writefile water_quality_app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier  # Demo model

# Demo model (replace with your trained one)
@st.cache_resource
def load_model():
    # Train dummy on sample Kenya data
    np.random.seed(42)
    X = np.random.rand(100, 4) * [14, 2000, 100, 50]  # ph, cond, turb, nit
    y = (X[:,0] > 6.5).astype(int)  # Safe if pH ok
    model = RandomForestClassifier()
    model.fit(X, y)
    joblib.dump(model, 'cloud_WaterQuality_model.keras')
    return model

model = load_model()

st.set_page_config(page_title="Water Quality", layout="wide")
st.markdown('<style>.main {background-color: #0e1117;}.metric {color: #00d4aa;}</style>', unsafe_allow_html=True)

st.title(" Kenya Water Quality Monitor")
st.sidebar.header("IoT Sensors")
ph = st.sidebar.slider("pH", 0.0, 14.0, 7.2)
turb = st.sidebar.slider("Turbidity", 0.0, 100.0, 5.2)
cond = st.sidebar.slider("Conductivity", 0.0, 2000.0, 280.0)
nit = st.sidebar.slider("Nitrates", 0.0, 50.0, 12.3)

input_data = np.array([[ph, cond, turb, nit]])
pred = model.predict(input_data)[0]
quality = "Safe 🟢" if pred else "Unsafe 🔴"

col1, col2, col3 = st.columns(3)
col1.metric("pH", f"{ph:.1f}")
col2.metric("Turbidity", f"{turb:.1f}")
col3.metric("Quality", quality)

fig = px.line(x=['Current'], y=[pred], title="Prediction Trend")
st.plotly_chart(fig)


Overwriting water_quality_app.py


In [65]:
from pyngrok import ngrok
import subprocess
import time
import threading

# Kill old tunnels
ngrok.kill()

# Set your ngrok authtoken here. Replace 'YOUR_AUTH_TOKEN' with your actual token.
# You can get a free token from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token('3BQMzaf7h8LHR5cMcfh6QY7iKxo_55uy9dmPfqh3ScaCfKLMg') # <----- ADD YOUR AUTH TOKEN HERE

# Background Streamlit
def run_streamlit():
    subprocess.Popen(["streamlit", "run", "water_quality_app.py", "--server.port", "8501", "--server.headless", "true"])

threading.Thread(target=run_streamlit).start()
time.sleep(5)  # Wait startup

# Ngrok tunnel (free, no account needed sometimes)
public_url = ngrok.connect(8501)
print("🚀 PUBLIC URL:", public_url)

🚀 PUBLIC URL: NgrokTunnel: "https://unplucked-westerly-charlott.ngrok-free.dev" -> "http://localhost:8501"


In [66]:
!npm install -g localtunnel
!streamlit run water_quality_app.py &>/content/logs.txt &
!npx localtunnel --port 8501


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 6s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://tough-books-like.loca.lt
^C
